### codes of RKL

#### 离散分类分布：精确枚举实现

- PyTorch 的 KLDivLoss/F.kl_div 约定是：input 是 log-space；默认 log_target=False 时，逐点计算的是 target * (target.log() - input)；reduction="batchmean" 才对应数学上的 batch KL 缩放。
    - $D_{\mathrm{KL}}(q_\theta \| p)=\sum_x q_\theta(x)\left[\log q_\theta(x)-\log p(x)\right]$

In [2]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)

B, K = 4, 6
q_logits = torch.randn(B, K, requires_grad=True)

# 固定目标分布 p；这里用随机 logits 构造一个合法概率分布
p_logits = torch.randn(B, K)
p_probs = F.softmax(p_logits, dim=-1).detach()


def rkl_manual(q_logits, p_probs, eps=1e-8):
    log_q = F.log_softmax(q_logits, dim=-1)          # [B, K]
    q = log_q.exp()                                  # [B, K]
    log_p = p_probs.clamp_min(eps).log()             # [B, K]

    # KL(q || p) = sum_x q(x) [log q(x) - log p(x)]
    return (q * (log_q - log_p)).sum(dim=-1).mean()


def rkl_torch(q_logits, p_probs, eps=1e-8):
    log_q = F.log_softmax(q_logits, dim=-1)
    q = log_q.exp()
    log_p = p_probs.clamp_min(eps).log()

    # F.kl_div(input, target):
    # target * (target.log() - input)
    # input = log_p, target = q  =>  KL(q || p)
    return F.kl_div(
        input=log_p,
        target=q,
        reduction="batchmean",
        log_target=False,
    )


loss_manual = rkl_manual(q_logits, p_probs)
loss_torch = rkl_torch(q_logits, p_probs)

grad_manual, = torch.autograd.grad(loss_manual, q_logits, retain_graph=True)
grad_torch, = torch.autograd.grad(loss_torch, q_logits)

print("loss_manual:", loss_manual.item())
print("loss_torch: ", loss_torch.item())
print("loss close: ", torch.allclose(loss_manual, loss_torch, atol=1e-7))

print("max grad diff:", (grad_manual - grad_torch).abs().max().item())
print("grad close:   ", torch.allclose(grad_manual, grad_torch, atol=1e-7))

loss_manual: 0.8777908682823181
loss_torch:  0.8777908682823181
loss close:  True
max grad diff: 1.4901161193847656e-08
grad close:    True


#### 连续分布：Monte Carlo + rsample

$$
D_{\mathrm{KL}}(q_\theta \| p)\approx\frac{1}{M}\sum_{i=1}^M\left[\log q_\theta(x_i)-\log p(x_i)\right],\quad x_i\sim q_\theta
$$

PyTorch 文档里也区分了 score-function estimator 和 pathwise derivative estimator；rsample() 用重参数化技巧，让样本可以参与反向传播。Normal 分布支持 has_rsample=True 和 rsample()。

- 如果 q 和 p 都是高斯，优先用解析 KL
    - $q_\theta(x)=\mathcal{N}(\mu_q,\sigma_q^2I)$
    - $p(x)=\mathcal{N}(\mu_p,\sigma_p^2I)$
    - $D_{\mathrm{KL}}(q\|p)
=
\log\frac{\sigma_p}{\sigma_q}
+
\frac{
\sigma_q^2+(\mu_q-\mu_p)^2
}{
2\sigma_p^2
}
-\frac{1}{2}$

In [3]:
import torch
from torch.distributions import Normal, Independent

def reverse_kl_diag_gaussian_mc(mu, log_std, p_dist, n_samples=8):
    """
    q_theta = diagonal Gaussian
    mu:      [B, D]
    log_std: [B, D]
    p_dist:  一个支持 log_prob(x) 的目标分布
             例如 Independent(Normal(...), 1)

    return: scalar
    """
    std = log_std.exp()

    # Independent(..., 1) 表示最后一维 D 是同一个事件的维度
    q_dist = Independent(Normal(mu, std), 1)

    # x: [M, B, D]
    x = q_dist.rsample((n_samples,))

    # log_q: [M, B]
    log_q = q_dist.log_prob(x)

    # log_p: [M, B]
    log_p = p_dist.log_prob(x)

    # Monte Carlo estimate of KL(q || p)
    rkl = log_q - log_p                         # [M, B]

    return rkl.mean()

- 让 $q_\theta$ 靠近标准正态分布 $p=\mathcal{N}(0,I)$

In [4]:
B, D = 32, 4

mu = torch.randn(B, D, requires_grad=True)
log_std = torch.zeros(B, D, requires_grad=True)

p_dist = Independent(
    Normal(torch.zeros(B, D), torch.ones(B, D)),
    1,
)

loss = reverse_kl_diag_gaussian_mc(mu, log_std, p_dist, n_samples=16)

loss.backward()

print(loss.item())
print(mu.grad.shape)
print(log_std.grad.shape)

1.4822486639022827
torch.Size([32, 4])
torch.Size([32, 4])


### why RKL not FKL

> **forward KL 不是"算不了"，是"算得出但没法用"**。从同样两个标量确实能构造一个无偏估计量，只是它的方差在 forward KL 最该发力的地方发散。这个区别很重要，因为它解释了为什么 verl 非要多传 64 个 logits。

> 还有一层期望，比 token 层更根本

上面讲的都是**单个位置内层的 next-token 期望**。